In [ ]:
# Imports
import tensorflow as tf
import pandas as pd
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.models import Sequential
import os
import numpy as np
from mealpy import FloatVar, GWO
import matplotlib.pyplot as plt
import datetime

np.random.seed(42)

In [ ]:
# Import train, val, and test data from preprocessing.ipynp

train_df = pd.read_csv("../working/train.csv")
val_df = pd.read_csv("../working/val.csv")
test_df = pd.read_csv("../working/test.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../working/val.csv'

In [ ]:
# data augmentation layers in the model
data_augmentation = Sequential([
    layers.RandomBrightness(factor=0.2,  value_range=(0, 255)),
    layers.RandomFlip(mode='horizontal_and_vertical'),
    layers.RandomRotation(factor=0.2)
], name='data_augmentation')

In [ ]:
# turn train, val, and test dataframes to images for model input
# converts images to tensor data structure
def make_ds(df, shuffle=True, batch=18):
    ds = tf.data.Dataset.from_tensor_slices((df['path'].values, df['diagnosis'].values))
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    def load(path, label):
        img = tf.io.decode_png(tf.io.read_file(path), channels=3)
        return tf.cast(img, tf.float32), label   # keep 0–255 values
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df), make_ds(test_df)

In [ ]:
# Pretrained MobileNetV3Small feature extractor
base = tf.keras.applications.MobileNetV3Small(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained layers
base.trainable = False

In [ ]:

# randomly choose values for dropout and dense below between values in paper table 5
dropout1 = np.random.uniform(0.2, 0.5)
dropout2 = np.random.uniform(0.2, 0.5)
dense1 = np.random.randint(128, 256)
dense2 = np.random.randint(32, 64)

# Build the complete model
model = Sequential([
    layers.Input(shape=(224, 224, 3)),

    # Data augmentation
    data_augmentation,

    # Feature extraction
    base, # produces (7, 7, 960) because we have include_top = False

    # Classification head
    layers.GlobalAveragePooling2D(data_format='channels_last', keepdims=False),
    layers.Dropout(dropout1),
    layers.Dense(dense1, activation="relu"),
    layers.Dropout(dropout2),
    layers.Dense(dense2, activation='relu'),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy",
         tf.keras.metrics.AUC(name="auc"),
         tf.keras.metrics.Precision(name="precision"),
         tf.keras.metrics.Recall(name="recall")]
)

# callback functions
cbs = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5,
                                     restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("../output/checkpoints/vgg16_f3_head.weights.h5", monitor="val_auc",
                                       mode="max", save_best_only=True, save_weights_only=True)
]

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,780,481 (56.38 MB)

 Trainable params: 65,793 (257.00 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
# train non GWO model
history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=cbs)

# save the model to the output directory
model.save('../output/models/nonGWO_model.h5')

Epoch 1/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 680s 3s/step - accuracy: 0.6620 - auc: 0.7084 - loss: 0.8137 - precision: 0.6647 - recall: 0.6536 - val_accuracy: 0.7718 - val_auc: 0.8651 - val_loss: 0.5238 - val_precision: 0.9005 - val_recall: 0.7235
Epoch 2/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 1660s 8s/step - accuracy: 0.7644 - auc: 0.8300 - loss: 0.5138 - precision: 0.7571 - recall: 0.7787 - val_accuracy: 0.8008 - val_auc: 0.8944 - val_loss: 0.4632 - val_precision: 0.9121 - val_recall: 0.7624
Epoch 3/30
104/198 ━━━━━━━━━━━━━━━━━━━━ 5:26 3s/step - accuracy: 0.7826 - auc: 0.8632 - loss: 0.4615 - precision: 0.7927 - recall: 0.7774

KeyboardInterrupt: 

## GWO hyperparameter search (Heikal et al. 2024)

Grey Wolf Optimization searches for the head's hyperparameters (learning rate, dropout, dense neurons, epochs) that give the highest **validation** accuracy, using the paper's search ranges and 10 iterations.

To make this feasible on a CPU, VGG16's output is computed **once** per image and cached. Since VGG16 is frozen during head training, every trial can then train only the small head on the cached 512-number feature vectors (seconds per trial instead of hours).

Built on the GWO code in `gwo.ipynb`.

In [ ]:
# GWO step 1: cache frozen-VGG16 features (runs once, then loads from disk)

feature_path = "../output/features/vgg16_f3_features.npz"

if os.path.exists(feature_path):
    data = np.load(feature_path)
    X_tr, y_tr, X_val, y_val = data["X_tr"], data["y_tr"], data["X_val"], data["y_val"]
else:
    # A fresh frozen MobileNetV3Small with the same
    # preprocessing and pooling as the full model
    feature_extractor = tf.keras.Sequential([
        layers.Input(shape=(224, 224, 3)),
        
        # Data augmentation
        data_augmentation,
    
        # Feature extraction
        base, # produces (7, 7, 960) because we have include_top = False
    ], name="mobilenet_features")

    def extract_features(df):
        # the upsampled train set repeats images: run each unique image once, then copy rows
        unique = df.drop_duplicates("path").reset_index(drop=True)
        feats = feature_extractor.predict(make_ds(unique), verbose=1)
        lookup = dict(zip(unique["path"], feats))
        X = np.stack([lookup[p] for p in df["path"]])
        y = df["diagnosis"].values.astype("float32")
        return X, y

    X_tr, y_tr = extract_features(train_df)
    X_val, y_val = extract_features(val_df)      # test set deliberately not touched
    np.savez(feature_path, X_tr=X_tr, y_tr=y_tr, X_val=X_val, y_val=y_val)

print("train features:", X_tr.shape, " val features:", X_val.shape)

In [ ]:
# GWO step 2: define the search and run it

def build_head(learning_rate, dropout1_rate, dropout2_rate, neurons1, neurons2):
    # same classification head as the full model, but its input is the cached 512-feature vector
    head = tf.keras.Sequential([
        layers.Input(shape=(7, 7, 960)),
        layers.GlobalAveragePooling2D(data_format='channels_last', keepdims=False),

        layers.Dropout(dropout1_rate),
        layers.Dense(neurons1, activation="relu"),
        layers.Dropout(dropout2_rate),
        layers.Dense(neurons2, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    head.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),
                 loss="binary_crossentropy", metrics=["accuracy"])
    return head

trial_log = []   # every hyperparameter set tried (for the results table)
cache = {}       # skip retraining identical sets (GWO's final iteration often repeats one)

def objective_function(solution):
    # each wolf's position is 4 continuous numbers -> convert to usable hyperparameters
    learning_rate = float(solution[0])
    epochs = int(round(solution[1]))
    dropout1_rate = float(solution[2])
    neurons1 = int(round(solution[3]))
    dropout2_rate = float(solution[4])
    neurons2 = int(round(solution[5]))

    # if you randomly end up with the same parameters, jsut return your previous validation accuracy
    key = (round(learning_rate, 6), round(dropout1_rate, 3), round(dropout2_rate, 3), neurons1, neurons2, epochs)
    if key in cache:
        return cache[key]

    tf.keras.backend.clear_session()     # free the previous trial's model
    tf.keras.utils.set_random_seed(42)   # same starting weights every trial -> fair comparison

    head = build_head(learning_rate, dropout1_rate, dropout2_rate, neurons1, neurons2)
    head.fit(X_tr, y_tr, epochs=epochs, verbose=0)
    _, val_acc = head.evaluate(X_val, y_val, verbose=0) # get the models performance with the given parameters

    # logging things
    cache[key] = float(val_acc)
    trial_log.append(dict(learning_rate=learning_rate, dropout1=dropout1_rate, dropout2=dropout2_rate,
                          neurons1=neurons1, neurons2=neurons2, epochs=epochs, val_acc=val_acc))
    
    return float(val_acc)

# search ranges from Heikal et al. 2024: [learning rate, dropout, dense neurons, epochs]
problem = {
    "bounds": FloatVar(lb=(1e-4, 5, 0.2, 128, 0.2, 32),
                       ub=(5e-4, 30, 0.5, 256, 0.5, 64),
                       name="hyperparameters"),
    "minmax": "max", # maximize validation accuracy (the paper's fitness)
    "obj_func": objective_function
}

# 10 iterations as in the paper, 100 wolves as in the paper
optimizer = GWO.OriginalGWO(epoch=5, pop_size=100)
best = optimizer.solve(problem, seed=42)

In [ ]:
# GWO results

best_lr = float(best.solution[0])
best_epochs = int(round(best.solution[1]))
best_dropout1_rate = float(best.solution[2])
best_neurons1 = int(round(best.solution[3]))
best_dropout2_rate = float(best.solution[4])
best_neurons2 = int(round(best.solution[5]))

print(f"GWO best: lr={best_lr:.6f}, dropout1_rate={best_dropout1_rate:.3f}, dropout2_rate={best_dropout1_rate:.3f}, neurons1={best_neurons1}, neurons2={best_neurons1}, epochs={best_epochs}")
print(f"GWO best validation accuracy: {best.target.fitness:.4f}")

# convergence: best accuracy found so far after each GWO iteration
plt.plot(range(1, len(optimizer.history.list_global_best_fit) + 1),
         optimizer.history.list_global_best_fit, marker="o")
plt.xlabel("GWO iteration"); plt.ylabel("best validation accuracy"); plt.title("GWO convergence")
plt.show()

results = pd.DataFrame(trial_log).sort_values("val_acc", ascending=False)
results.to_csv("../output/gwo_trials_f3.csv", index=False)
results.head(10)

In [ ]:
# GWO step 4: train the full model (with augmentation) using the GWO hyperparameters
gwo_augmentation = Sequential([
    layers.RandomBrightness(factor=0.2, value_range=(0, 255)),
    layers.RandomFlip(mode="horizontal_and_vertical"),
    layers.RandomRotation(factor=0.2),
], name="gwo_augmentation")

gwo_model = Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    base,
    layers.GlobalAveragePooling2D(data_format="channels_last", keepdims=False)
    layers.Dropout(best_dropout1_rate),
    layers.Dense(best_neurons1, activation="relu"),
    layers.Dropout(best_dropout2_rate),
    layers.Dense(best_neurons2, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

gwo_model.compile(optimizer=tf.keras.optimizers.Adam(best_lr),
                loss="binary_crossentropy",
                metrics=["accuracy", 
                    tf.keras.metrics.AUC(name="auc"),
                    tf.keras.metrics.Precision(name="precision"),
                    tf.keras.metrics.Recall(name="recall")
                ])

gwo_ckpt = tf.keras.callbacks.ModelCheckpoint("../output/vgg16_f3_gwo.weights.h5", monitor="val_accuracy",
                                              mode="max", save_best_only=True, save_weights_only=True)

# epochs is one of the GWO-tuned hyperparameters, so no early stopping here
history_gwo = gwo_model.fit(train_ds, validation_data=val_ds, epochs=best_epochs, callbacks=[gwo_ckpt])

gwo_model.save('../output/models/GWO_model.h5')